In [ ]:
#always run this first!!
#essential reticulate functions that allow us to use python packages in R
#you'll need a conda env with 'leidenalg' and 'pandas' installed to do this
#then route reticulate to the python installed in that conda env with the below functions
Sys.setenv(RETICULATE_PYTHON="/user/.conda/envs/reticulate/bin/python")
reticulate::use_python("/user/.conda/envs/reticulate/bin/python")
reticulate::use_condaenv("/user/.conda/envs/reticulate")
reticulate::py_module_available(module='leidenalg') #needs to be TRUE
reticulate::import('leidenalg') #good to make sure this doesn't error

suppressMessages(library(GenomicRanges))
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(EnsDb.Hsapiens.v86))
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(Matrix))
suppressMessages(library(harmony))
suppressMessages(library(data.table))
suppressMessages(library(ggpubr))
suppressMessages(library(tidyverse))
suppressMessages(library(knitr))
suppressMessages(library(SoupX))
suppressMessages(library(plyr))

suppressMessages(require(DoubletFinder))
opts_chunk$set(tidy=TRUE)
warnLevel <- getOption('warn')
options(warn = -1)

# Create a raw merged object

### Read in singlome rds previously created

In [ ]:
samples <- c('MM_354','MM_355','MM_356','MM_357','MM_358','MM_359','MM_360','MM_380','MM_381','MM_382','MM_398','MM_399',
           'MM_401','MM_402','MM_403','MM_404','MM_405','MM_406','MM_408','MM_361','MM_407','MM_400','MM_426','MM_397',
           'MM_553','MM_554','MM_555','MM_556','MM_557','MM_558','MM_559','MM_560','MM_561')

In [ ]:
list_adata <- list()
for (sample in samples){
    print(paste(sample, Sys.time()))
    #read in RDS
    list_adata[sample] <- readRDS(file = sprintf('/data/%s_postqc_doubletFinder_soupXgeneSelect.rds', sample))
    
    }

In [ ]:
npod_combine <- merge(list_adata$MM_354, y = c(list_adata$MM_355,list_adata$MM_356,list_adata$MM_357,
                                               list_adata$MM_358,list_adata$MM_359,list_adata$MM_360,list_adata$MM_380,
                                               list_adata$MM_381,list_adata$MM_382,list_adata$MM_398,list_adata$MM_399,
                                               list_adata$MM_401,list_adata$MM_402,list_adata$MM_403,list_adata$MM_404,
                                               list_adata$MM_405,list_adata$MM_406,list_adata$MM_408,list_adata$MM_361,
                                               list_adata$MM_407,list_adata$MM_400,list_adata$MM_426,list_adata$MM_397,
                                               list_adata$MM_553,list_adata$MM_554,list_adata$MM_555,list_adata$MM_556,
                                               list_adata$MM_557,list_adata$MM_558,list_adata$MM_559,list_adata$MM_560,
                                               list_adata$MM_561), 
                      add.cell.ids = c('MM_354','MM_355','MM_356','MM_357',
           'MM_358','MM_359','MM_360','MM_380',
           'MM_381','MM_382','MM_398','MM_399',
           'MM_401','MM_402','MM_403','MM_404',
           'MM_405','MM_406','MM_408','MM_361',
           'MM_407','MM_400','MM_426','MM_397',
           'MM_553','MM_554','MM_555','MM_556',
           'MM_557','MM_558','MM_559','MM_560',
           'MM_561'), project = "NPOD")
npod_combine@meta.data$technology <- 'snRNA'
npod_combine


In [ ]:
saveRDS(npod_combine, file = '/data/npod_singlome_doubletFinder_soupXgeneSelect_mergeUnprocessed.rds')



In [ ]:
npod_singlome <- readRDS(file = '/data/npod_singlome_doubletFinder_soupXgeneSelect_mergeUnprocessed.rds')

### Read in multiome data

In [ ]:
samples <- c('MM_662','MM_664','MM_667','MM_666','MM_665','MM_660','MM_661','6229_sort')

In [ ]:
list_adata <- list()
for (sample in samples){
    print(paste(sample, Sys.time()))
    
    #read in RDS
    list_adata[sample] <- readRDS(file = sprintf('/data/%s_postqc_doubletFinder_soupXgeneSelect.rds', sample))
    
    }

In [ ]:
npod_combine_multiome <- merge(list_adata$MM_662, y = c(list_adata$MM_664,list_adata$MM_667,list_adata$MM_666,
                                               list_adata$MM_665,list_adata$MM_660,list_adata$MM_661,list_adata$`6229_sort`), 
                      add.cell.ids = c('MM_662','MM_664','MM_667','MM_666','MM_665','MM_660','MM_661','MM_510'), project = "NPOD")
npod_combine_multiome@meta.data$technology <- 'multiome'

npod_combine_multiome

In [ ]:
saveRDS(npod_combine_multiome, file = '/data/npod_multiome_doubletFinder_soupXgeneSelect_mergeUnprocessed_rndInt.rds')


In [ ]:
npod_multiome <- readRDS(file = '/data/npod_multiome_doubletFinder_soupXgeneSelect_mergeUnprocessed_rndInt.rds')

## Merge singlome and multiome_RNA objects

In [ ]:
npod_combine <- merge(npod_singlome, y=npod_multiome)

In [ ]:
#### add metadata
npod_combine$samples <- substr(rownames(npod_combine@meta.data),1,6)

sex_F = list('MM_660','MM_663','MM_664','MM_662','MM_510','MM_667','MM_358','MM_361','MM_380','MM_382','MM_398','MM_400','MM_401','MM_402','MM_403','MM_404','MM_405','MM_406','MM_407','MM_426','MM_559','MM_560','MM_561')
cond_t1d = list('MM_665','MM_666','MM_661','MM_662','MM_354','MM_355','MM_358','MM_361','MM_380','MM_397','MM_398','MM_401','MM_402','MM_405','MM_406','MM_555')
cond_aab = list('MM_667','MM_357','MM_399','MM_403','MM_408','MM_553','MM_556','MM_557','MM_559','MM_560')
cond_ctrl = list('MM_660','MM_663','MM_664','MM_510','MM_356','MM_359','MM_360','MM_381','MM_382','MM_400','MM_404','MM_407','MM_426','MM_554','MM_558','MM_561')

npod_combine@meta.data$sex[npod_combine@meta.data$samples %in% sex_F] <- 'F'
npod_combine@meta.data$sex[!npod_combine@meta.data$samples %in% sex_F] <- 'M'

npod_combine@meta.data$condition[npod_combine@meta.data$samples %in% cond_t1d] <- 'T1D'
npod_combine@meta.data$condition[npod_combine@meta.data$samples %in% cond_aab] <- 'Aab'
npod_combine@meta.data$condition[npod_combine@meta.data$samples %in% cond_ctrl] <- 'Control'
head(npod_combine@meta.data)


In [ ]:
#### save unprocessed object,in order to not repeat the above steps in the future
saveRDS(npod_combine, file = '/data/npod_merge_doubletFinder_soupXgeneSelect_mergeProcessed_rndInt.rds')


In [ ]:
#### optional, restart notebook and read in this object to clear up memory usage
npod_combine <- readRDS('/data/npod_merge_doubletFinder_soupXgeneSelect_mergeProcessed_rndInt.rds')

### Process merged object

In [ ]:
npod_combine <- NormalizeData(npod_combine, normalization.method = "LogNormalize", scale.factor = 10000)

npod_combine <- FindVariableFeatures(npod_combine, selection.method = "vst", nfeatures = 2000)
npod_combine <- ScaleData(npod_combine, verbose = FALSE) %>% 
    RunPCA(pc.genes = npod_combine@var.genes, npcs = 50, verbose = FALSE)


options(repr.plot.height = 2.5, repr.plot.width = 6)
npod_combine <- npod_combine %>% 
    RunHarmony(c("samples","sex","technology"), plot_convergence = TRUE)


npod_combine <- npod_combine %>% 
    RunUMAP(reduction = "harmony", dims = 1:50) %>% 
    FindNeighbors(reduction = "harmony", dims = 1:50) %>% 
    FindClusters(algorithm=4,resolution = 0.5,method = "igraph") 


#options(repr.plot.height = 4, repr.plot.width = 6)
DimPlot(npod_combine, reduction = "umap", label = TRUE, pt.size = .1)
#1,2,3,4,14
p4 <- DimPlot(npod_combine, reduction = "umap", label = TRUE, pt.size = .1)
print(p4)
marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
              'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
              'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
              'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
              'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
              'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
              'NCR1','NCAM1','CD4','FOXP3')
#options(repr.plot.width=12, repr.plot.height=6)
p5 <- DotPlot(npod_combine, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
p5 <- p5 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
print(p5)

In [ ]:
options(repr.plot.width=12, repr.plot.height=6)

DimPlot(npod_combine, reduction = "umap", label = TRUE, pt.size = .1)
#1,2,3,4,14
p4 <- DimPlot(npod_combine, reduction = "umap", label = TRUE, pt.size = .1)
print(p4)
marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
              'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
              'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
              'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
              'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
              'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
              'NCR1','NCAM1','CD4','FOXP3')
#options(repr.plot.width=12, repr.plot.height=6)
p5 <- DotPlot(npod_combine, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
p5 <- p5 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
print(p5)

In [ ]:
options(repr.plot.width=20, repr.plot.height=16)

FeaturePlot(npod_combine, features = c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','REG1A'))



In [ ]:
#### save processed object so you don't have to repeat the steps
npod_combine<- readRDS(file = '/data/npod_singlome_doubletFinder_soupXgeneSelect_mergeProcessed.rds')


In [ ]:
options(repr.plot.width=10, repr.plot.height=15)
p1 <- VlnPlot(npod_combine, features='nFeature_RNA',  pt.size=0, log=FALSE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(npod_combine$nFeature_RNA), linetype='dashed')#
p2 <- VlnPlot(npod_combine, features='nCount_RNA', pt.size=0, log=FALSE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept = 7500)#geom_hline(yintercept=median(npod_combine$nCount_RNA), linetype='dashed')
p3 <- VlnPlot(npod_combine, features='percent.mt',  pt.size=0, log=FALSE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(npod_combine$percent.mt), linetype='dashed')
p1 / p2 / p3


### Filter out low quality cells

In [ ]:
#### filter out cells with high mitochondrial content, reads and genes expressed
filter <- subset(npod_combine,subset = percent.mt <= 1)
filter
filter <- subset(filter,subset = nFeature_RNA < 4000)
filter
filter <- subset(filter,subset = nCount_RNA < 7500)
filter

In [ ]:
options(repr.plot.width=10, repr.plot.height=15)
p1 <- VlnPlot(filter, features='nFeature_RNA',  pt.size=0, log=FALSE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(filter$nFeature_RNA), linetype='dashed')#
p2 <- VlnPlot(filter, features='nCount_RNA', pt.size=0, log=FALSE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept = 7500)#geom_hline(yintercept=median(npod_combine$nCount_RNA), linetype='dashed')
p3 <- VlnPlot(filter, features='percent.mt',  pt.size=0, log=FALSE) + geom_boxplot(width=.6, fill='white', alpha=.6) + geom_hline(yintercept=median(filter$percent.mt), linetype='dashed')
p1 / p2 / p3


In [ ]:
#### recluster object
options(repr.plot.width=12, repr.plot.height=6)

filter[['percent.mt']] <- PercentageFeatureSet(filter, pattern = '^MT-')
filter <- NormalizeData(filter, normalization.method = "LogNormalize", scale.factor = 10000)

filter <- FindVariableFeatures(filter, selection.method = "vst", nfeatures = 2000)
all.genes <- rownames(filter)

filter <- ScaleData(filter, verbose = FALSE) %>% 
    RunPCA(pc.genes = filter@var.genes, npcs = 50, verbose = FALSE)


#options(repr.plot.height = 2.5, repr.plot.width = 6)
filter <- filter %>% 
    RunHarmony(c("samples","sex","technology"), plot_convergence = TRUE)


filter <- filter %>% 
    RunUMAP(reduction = "harmony", dims = 1:50) %>% 
    FindNeighbors(reduction = "harmony", dims = 1:50) %>% 
    FindClusters(algorithm=4,resolution = 0.5,method = "igraph") 


#options(repr.plot.height = 4, repr.plot.width = 6)
DimPlot(filter, reduction = "umap", label = TRUE, pt.size = .1)

p4 <- DimPlot(filter, reduction = "umap", label = TRUE, pt.size = .1)
print(p4)
marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
              'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
              'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
              'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
              'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
              'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
              'NCR1','NCAM1','CD4','FOXP3')
#options(repr.plot.width=12, repr.plot.height=6)
p5 <- DotPlot(filter, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
p5 <- p5 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
print(p5)

options(repr.plot.width=20, repr.plot.height=16)

FeaturePlot(filter, features = c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','REG1A'))



In [ ]:
### write out list of barcodes in any low quality clusters
lowqual_clusters = c( 20,23,24,23,26, 27)
outdir2 <-'/data/RNA_remove_cells/'
to_remove <- file.path(outdir2,"lowqual1.txt")
write.table(filter[[]][filter$seurat_clusters %in% lowqual_clusters, c()], 
            to_remove, col.names=FALSE, quote=FALSE)

### subcluster any clusters to identify additional doublets manually

In [ ]:
#Subclustering workaround #1: pass the 'igraph' method to leidenalg
FindSubCluster1 <- function(
  object,
  cluster,
  graph.name,
  subcluster.name = "sub.cluster",
  resolution = 0.5,
  algorithm = 1
) {
  sub.cell <- WhichCells(object = object, idents = cluster)
  sub.graph <- as.Graph(x = object[[graph.name]][sub.cell, sub.cell])
  sub.clusters <- FindClusters(
    object = sub.graph,
    resolution = resolution,
    algorithm = algorithm,
    method = 'igraph'
  )
  sub.clusters[, 1] <- paste(cluster,  sub.clusters[, 1], sep = "_")
  object[[subcluster.name]] <- as.character(x = Idents(object = object))
  object[[subcluster.name]][sub.cell, ] <- sub.clusters[, 1]
  return(object)
}

In [ ]:
sc.num <- 19

data_subcluster <- FindSubCluster1(filter, cluster=sc.num, algorithm=4, resolution=1.5, graph.name = 'RNA_snn') 

options(repr.plot.width=10, repr.plot.height=8)
data_subset <- subset(data_subcluster, subset=seurat_clusters==sc.num)
p1 <- DimPlot(data_subset, reduction='umap', group.by='sub.cluster', label=TRUE, label.size=6, repel=TRUE)
p1

# marker.genes <- c('CD69','C1QB', 'C1QC', 'C1QA', #Immune (macro),
#                   'CXCR4', 'GATA3', 'CD3D', 'CD4',#T-cell)
#                     'CHH19', 'S100V', 'CRYAB', 'PMP22', 'SCN7A') #Schwann

# marker.genes <- c('INS','IAPP','HADH', #Beta
#                   'GCG','FAP','TTR') #Alpha

marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
              'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
              'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
              'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
              'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
              'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
              'NCR1','NCAM1','CD4','FOXP3')

marker.genes <- unique(marker.genes)
FeaturePlot(data_subset, features = c('REG1A' ,'INS','CFTR'))

options(repr.plot.width=25, repr.plot.height=12)
p1 <- DotPlot(data_subset, assay='RNA', group.by='sub.cluster', features=marker.genes, cluster.idents=TRUE) 
p1 <- p1 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
p1

In [ ]:
### write out file with barcodes identified as doublets 
subclusters_to_remove = c('22_7','22_5')
write.table(data_subset[[]][data_subset$sub.cluster %in% subclusters_to_remove, c()], 
            '/data/RNA_remove_cells/1nov_npod_clust22.doublet1',col.names=FALSE, quote=FALSE)


In [ ]:
#number of cells in each cluster
table(filter[[]]$seurat_clusters)

#barplot of distribution of samples in each cluster
options(repr.plot.width=25, repr.plot.height=5)
theme_set(
    theme_classic())
filter$value <- 1
p2 <- ggplot(filter[[]], aes(fill=samples, y=value, x=seurat_clusters)) + 
geom_bar(position=position_fill(reverse=TRUE), stat='identity') + xlab('') + 
ylab('Percentage') + NoLegend() +
theme(plot.title=element_text(hjust=0.5,size=25),
      axis.title=element_text(size=30),
      axis.text=element_text(size=25),axis.ticks=element_blank())
p2

### remove barcodes previously flagged as low quality

In [ ]:
to_removeFinal <-  file.path(outdir2,"removal_1_doublets.txt")
to_removeFinal2 <-file.path(outdir2,"lowqual1.txt")

remove.cells1 <- trimws(scan(to_removeFinal, what='', sep='\n'), which='right', whitespace=' ')
remove.cells2 <- trimws(scan(to_removeFinal2, what='', sep='\n'), which='right', whitespace=' ')

remove <- c(remove.cells1,remove.cells2)
print(length(remove))
filter$remove_cells <- (Cells(filter) %in% remove)
filter2 <- subset(filter, subset=remove_cells==FALSE)


In [ ]:
### recluster 
options(repr.plot.width=12, repr.plot.height=6)

filter2[['percent.mt']] <- PercentageFeatureSet(filter2, pattern = '^MT-')
filter2 <- NormalizeData(filter2, normalization.method = "LogNormalize", scale.factor = 10000)

filter2 <- FindVariableFeatures(filter2, selection.method = "vst", nfeatures = 2000)
all.genes <- rownames(filter2)

filter2 <- ScaleData(filter2, verbose = FALSE) %>% 
    RunPCA(pc.genes = filter2@var.genes, npcs = 50, verbose = FALSE)


#options(repr.plot.height = 2.5, repr.plot.width = 6)
filter2 <- filter2 %>% 
    RunHarmony(c("samples","sex","technology"), plot_convergence = TRUE)


filter2 <- filter2 %>% 
    RunUMAP(reduction = "harmony", dims = 1:50) %>% 
    FindNeighbors(reduction = "harmony", dims = 1:50) %>% 
    FindClusters(algorithm=4,resolution = 0.5,method = "igraph") 


#options(repr.plot.height = 4, repr.plot.width = 6)
DimPlot(filter2, reduction = "umap", label = TRUE, pt.size = .1)

p4 <- DimPlot(filter2, reduction = "umap", label = TRUE, pt.size = .1)
print(p4)
marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
              'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
              'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
              'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
              'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
              'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
              'NCR1','NCAM1','CD4','FOXP3')
#options(repr.plot.width=12, repr.plot.height=6)
p5 <- DotPlot(filter2, assay='RNA', features=marker.genes, cluster.idents=TRUE) 
p5 <- p5 + theme(axis.text.x=element_text(angle=45, hjust=1)) + xlab('') + ylab('')
print(p5)

options(repr.plot.width=20, repr.plot.height=16)

FeaturePlot(filter2, features = c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','REG1A'))



In [ ]:
saveRDS(filter2, file = '{file_name}.rds')

In [ ]:
#### Assign celltype names 
filter2@meta.data$celltype_assignment = as.vector(mapvalues(filter2$seurat_clusters,from =  c('1','2','3','4','5','6','7','8','9','10','11','12','13','14','15','16','17','18','19','20','21','22','23'), 
         to = c('Acinar', 'Acinar',
                'Acinar','Ductal',
                'REG+Acinar','Acinar',
                'ActivatedStellate','Beta',
                'Alpha','Ductal',
                'Acinar','Macrophage',
                'Immune','Endothelial',
                'Delta','Stellate_q',
                'Ductal','Schwann',
               'Acinar','Acinar',
               'Acinar','Acinar','ActivatedStellate'), warn_missing = TRUE))



In [ ]:
p4 <- DimPlot(filter2, reduction = "umap", group.by = 'celltype_assignment',label = TRUE, pt.size = .1)
print(p4)

In [ ]:
table(data_subset$sub.cluster)

In [ ]:
subclusters_to_remove = c('8_12')
write.table(data_subset[[]][data_subset$sub.cluster %in% subclusters_to_remove, c()], 
            '/data/RNA_remove_cells/7nov_npod_clust8.doublet1',col.names=FALSE, quote=FALSE)


In [ ]:
FeaturePlot(data_subset, features = c('PPY' ,'INS','SST'))


In [ ]:
saveRDS(filter2, file = '{file_name}.rds')

#### Additional marker genes/plots

In [ ]:
mast.genes <- c('KIT','TPSAB1','RGS13','HDC','MS4A2','HPGDS')

immune.genes <- c('CD69','KLRB1','CD8B','IFNG')

bcell.genes <- c('MS4A1', 'TNFRSF13B', 'IGHM', 'IGHD', 'AIM2', 'CD79A', 'LINC01857', 
                 'RALGPS2', 'BANK1', 'CD79B','COCH', 'BANK1', 'SSPN', 'TEX9', 
                 'TNFRSF13C', 'LINC01781','IGHM', 'IGHD', 'IL4R', 'CXCR4', 
                 'BTG1', 'TCL1A', 'YBX3','IGHA2', 'MZB1', 'TNFRSF17', 'DERL3', 'TXNDC5', 
                 'TNFRSF13B', 'POU2AF1', 'CPNE5', 'NT5DC2')
bcell.genes <- unique(bcell.genes)

nkcell.genes <- c('NCR1','NCAM1','NKG7', 'KLRD1', 'TYROBP', 
                  'GNLY', 'FCER1G', 'PRF1', 'CD247', 'KLRF1', 'CST7', 'GZMB')

cd4.genes <-c('CD4','FOXP3','IL7R', 'MAL', 'LTB', 'LDHB', 
              'TPT1', 'TRAC', 'TMSB10', 'CD3D', 'CD3G')

cd8.genes <-c('CD8B', 'CD8A', 'CD3D', 'TMSB10', 'HCST', 
              'CD3G', 'LINC02446', 'CTSW', 'CD3E', 'TRAC',
              'IFNG','KLRB1')
                 
misct.genes <-c('CD3D', 'GZMK', 'KLRB1', 'NKG7', 
                'TRGC2', 'CST7', 'LYAR', 'KLRG1', 'GZMA')   
APC.genes <-   c('HLA-DPA1', 'HLA-DPB1', 'HLA-DQA1', 'HLA-DRA', 'HLA-DMA','HLA-DQB1', 'HLA-DRB1')              
dendritic.genes <-c('CD74', 'CCDC88A',  'CST3', 'CD11c')
                 
macrophage.genes <-c('CD74', 'PTPRC', 'ZEB2','CD14','CD68','CCR5')
marker.genes <- c('INS','GCG','SST','PPY','CTRB2','CFTR','PLVAP','COL1A1','C1QC','S100B',
                  'CD3D','ONECUT1','PDX1','ARX','MKI67','CDH19','CRYAB','PMP22','SCN7A',
                  'FLT4','PROX1','PDPN','LYVE1','MUC5B','TFF2','TFF3','CRISP3','CD1E',
                  'FLT3','CD1C','REG1A','REG3A','REG3G','REG1B','SPARCL1','FABP4','ITGA1','DES',
                  'COL5A2','COL6A3','LAMA2','LAMB1','SLIT2','LUM','KIT','TPSAB1','RGS13',
                  'HDC','MS4A2','HPGDS','CD69','KLRB1','CD8B','IFNG','CD19','CD79A','CD79B',
                  'NCR1','NCAM1','CD4','FOXP3')
gent.genes <- unique(paste(cd4.genes,cd8.genes,misct.genes))
gent.genes <- unique(unlist(strsplit(gent.genes, " ")))

immune.lists <-c(APC.genes, mast.genes,immune.genes,bcell.genes,nkcell.genes,
                 dendritic.genes,macrophage.genes,gent.genes, marker.genes)
immune.lists <-unique(immune.lists)
options(repr.plot.width=20, repr.plot.height=10)
p1 <- DotPlot(data_subset, assay='RNA', group.by='sub.cluster', features=immune.lists)#, cluster.idents=TRUE) 
p1 <- p1 + theme(axis.text.x=element_text(angle=90, hjust=1)) + xlab('') + ylab('')
p1

In [ ]:
unique(data_subset@meta.data$sub.cluster )
unique(data_subset@meta.data$seurat_clusters )

#### Assign subclusters celltype assignments

In [ ]:
sc.num <- 19

data_subcluster <- FindSubCluster1(filter, cluster=sc.num, algorithm=4, resolution=1.5, graph.name = 'RNA_snn') 

options(repr.plot.width=10, repr.plot.height=8)
data_subset <- subset(data_subcluster, subset=seurat_clusters==sc.num)
p1 <- DimPlot(data_subset, reduction='umap', group.by='sub.cluster', label=TRUE, label.size=6, repel=TRUE)
p1


In [ ]:
## FIRST TIME ONLY
#npod4@meta.data$seurat_clusters <- data_subcluster@meta.data$sub.cluster


###For remaining subclusters
for (x in 1:nrow(data_subcluster@meta.data)){
    if (data_subcluster@meta.data[x,]$seurat_clusters ==  13){
        npod4@meta.data[x,]$seurat_clusters <-  data_subcluster@meta.data[x,]$sub.cluster
    }    
}

In [ ]:
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "14_1"),]$celltype_assignment <- "Endothelial"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "14_2"),]$celltype_assignment <- "LymphEndo"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "14_3"),]$celltype_assignment <- "Endothelial"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "14_4"),]$celltype_assignment <- "Endothelial"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "4_13"),]$celltype_assignment <- "MUC5b_Ductal"

npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_1"),]$celltype_assignment <- "MastCells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_2"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_3"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_4"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_5"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_6"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_7"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_8"),]$celltype_assignment <- "Bcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_9"),]$celltype_assignment <- "Tcells"
npod4@meta.data[which(npod4@meta.data$seurat_clusters == "13_10"),]$celltype_assignment <- "MastCells"



p4 <- DimPlot(npod4, reduction = "umap", group.by = 'celltype_assignment',label = TRUE, pt.size = .1)
print(p4)

In [ ]:
saveRDS(npod4, file='20221106_npod4_labelled.rds')